# CEM4644 - MP4: Segmentation for a quantity take-off

## Homework (individual): *Seven sheets from three disciplines*

**No coding needed.** Each grey box below is one *step*: click the (play) button at its left, wait until it finishes,
look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 120 minutes)**
1. Look at the seven drawings: what each one is, what it shows, and what you have to take off from it.
2. Do a take-off on each discipline in turn - floor plans, structural plans, MEP plans - with the same one cell.
3. Try a drawing of your own.

**Before you start:** menu *Runtime -> Change runtime type -> T4 GPU -> Save*. The model used here (SAM 3) is large:
with a GPU each request takes well under a second; without one the steps that need the model take about a minute each.

Everything is in **feet and square feet**. There is no scale printed on a drawing that you can trust blindly: you set
the scale yourself, from a dimension the drawing prints or from something whose real size you know.

In [ ]:
#@title ▶ Step 0 · Run me first (2-3 minutes) { display-mode: "form" }
#@markdown Click the play button and wait for the 'Ready' line. This downloads the drawings with their answer keys and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the steps that need the live model.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"
FOLDERS = ["mp4_segmentation"]               # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="homework", load_model=load_model)


## Part 1 · The drawings

Seven real drawings: two floor plans (a 1940 farmhouse and a modern VA clinic), four structural foundation plans and one reflected ceiling plan. Each one carries a different take-off task - areas of rooms, areas of footings, counts of repeated symbols - and each one has an answer key.

You already met SAM 3 in the workshop, so this notebook goes straight to the work. What stays the same on every sheet:

- **you** set the scale, from a dimension the sheet prints or from something whose size the sheet tells you. Never from a
  scale bar you have not checked.
- **you** draw the boxes. The model turns a box into an outline; it does not know what a footing or a diffuser is.
- **counts come from the model, never from a tally of your own boxes**: where a sheet asks for a count you box ONE
  example of the symbol, labelled *example: ...*, and SAM 3 finds all the others like it.
- everything you measure and everything you count is checked against an answer key, so you always see how far off you are.

In [ ]:
#@title ▶ Step 1a · Browse the seven drawings { display-mode: "form" }
#@markdown *all drawings* shows all seven with the take-off tasks for each. Pick one drawing from the list to see it large before you work on it.
drawing = "all drawings" #@param ["all drawings", "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)", "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)", "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.show_sheets(drawing)


In [ ]:
#@title ▶ Step 1b · The symbol legend { display-mode: "form" }
#@markdown The MEP sheets carry a legend of their symbols: a bold rectangle with a diagonal and a small circle is a light fixture (2 ft x 4 ft or 2 ft x 2 ft), a small circle with a cross is a recessed light. The structural sheets name their footings in a schedule printed on the sheet instead.
drawing = "all" #@param ["all", "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)", "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)", "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.show_legend(drawing)


## Part 2 · The take-off, discipline by discipline

The same cell three times, with a different list of drawings. In each one: pick the drawing, then pick the label above
the picture before each box you draw (the labels come from that sheet's answer key), then *Submit*.

Three kinds of label: **scale: ...** for a length whose size the sheet gives you, the plain word (**room**, **footing**,
**pit**) for something you want the area of, and **example: ...** for something you want counted. One box labelled
*example: pile footing* is all a count needs: SAM 3 goes and finds every other symbol on the sheet that looks like it,
and that is how a count of 41 light fixtures or 29 pile footings is made. The slider under the picture sets how sure the
model has to be before it keeps one of them.

### Step 2a · Floor plans

On a floor plan you take off **areas of rooms**. Set the scale from a printed dimension, never from a scale bar: one of these two sheets carries a graphic scale bar that is wrong by a factor of two, and boxing it as well as the printed dimension is how you find that out. Then box every room the task asks for. The number you get is the *net* floor area: what the drawing puts on the floor (a counter, a bathtub) is cut out of the mask unless the room is a plain rectangle. The clinic sheet also asks for two **counts**, the water closets and the lavatories, and each one comes from a single box labelled *example: water closet* or *example: lavatory*.

- **usda_5542** (Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)): Set the scale: box the 32'-0" dimension at the top from arrowhead tip to arrowhead tip (label: scale: 32'-0" dimension). Box the 29'-0" dimension down the left side as well (label: scale: 29'-0" dimension) and compare the two readings. Box every room, porch, hall and closet (label: room). Careful: the sizes printed on this sheet do not agree with the drawing. The bedroom lettered 10' x 11' and the one lettered 11' x 11' are drawn in the same 10.4 ft wide column, and the BATH is lettered 5' x 7' but drawn 6.9 ft x 4.7 ft. The answer key is the area you measure off the drawing, not the lettered size; say in your report which rooms disagree and by how much.
- **va_floor** (VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)): Box the printed 10'-0" dimension on the top dimension string, arrowhead to arrowhead (label: scale: printed 10'-0" dimension). Box the 0-16 ft graphic bar at the top right, from its 0' mark to its 16' mark (label: scale: 0-16 ft graphic scale bar). The two scales will not agree - work out which one is wrong and by how much before you measure anything. Box at least six rooms (label: room). Their measured area is net floor area: the model cuts the furniture out of the mask, so expect to read about 10 % low in a furnished room and closer in an empty one. Box ONE water closet with the label 'example: water closet' and SAM 3 counts them all. Box ONE lavatory (the small wash basin next to a water closet) with the label 'example: lavatory' and count those the same way. This one is harder - read the hint.

In [ ]:
#@title ▶ Step 2a · Take-off on a floor plan { display-mode: "form" }
#@markdown Do **every** drawing in this list, one at a time: usda_5542, va_floor. Zoom with the mouse wheel. Copy each table into your report.
drawing = "usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)" #@param ["usda_5542: Five-room farmhouse, 32 ft x 29 ft (USDA design 710-5542)", "va_floor: VA outpatient clinic, lease module: floor plan (rooms and plumbing fixtures)"]
lab.takeoff(drawing)


> ### 📝 Report question 1
> From the floor plans: your scale reading on each sheet and how far it is from the answer key. On the VA clinic sheet you were asked to box the graphic scale bar as well as the printed dimension - what did the two give, and which one is right? (Work out what the areas would have been if you had trusted the bar.) Then the room table of one sheet: your square feet, the drawing's, the error. Which rooms are worst and why? Finally, the two counts SAM 3 made from your example boxes on the clinic sheet (found / missed / extra): which of the two would you hand on, and which would you check yourself first?

### Step 2b · Structural (foundation) plans

On a foundation plan you take off **areas of footings** and a **count of them**. The scale comes from a printed bay dimension or from a footing whose width its mark gives you (F4.0 = 4'-0" wide). The count is made by the model from one box labelled *example: footing* - you never count them yourself. Two things to watch: read the *same* edge of the ink at both ends (outside-to-outside or centre-to-centre, not one of each), and remember that a symbol smaller than about 60 pixels on the sheet is too small for the model - the mask becomes a rounded copy of your box, and the 'area' you get is the area you drew.

- **test_fp** (Foundation plan: 12 ft square spread footings around an equipment pit): This sheet prints no dimension and carries no scale bar. You are told instead: every square footing here is marked F12.0, which means 12'-0" x 12'-0". Box one of them (label: scale: spread footing, 12'-0" wide) and work the scale out from that. Box several footings (label: footing). Each one is 144 sq ft, so you can see straight away how close the model gets. Box ONE footing - an isolated one - with the label 'example: footing' and SAM 3 counts them all. There are 10. The thing in the middle is a pair of equipment pits, not a footing. No size is printed for it anywhere, so there is no area task on it.
- **test_fp_2** (Foundation plan with square spread footings and an elevator shaft): Box the printed 14'-0" bay X2-X3 on the bottom dimension string, tick to tick (label: scale: printed 14'-0" bay X2-X3). Box one heavy black F4.0 footing as a second scale (label: scale: F4.0 footing, 4'-0" wide) - it is 4'-0" wide - and compare the two answers. Box several footings (label: footing). The hexagon next to each one gives its size: E4'-6", E4'-8", E4'-10", E5'-0", and F4.0 for the two heavy black ones. Box the elevator shaft opening (label: pit). Box ONE grey footing with the label 'example: footing' and SAM 3 counts them all. There are 19: 17 grey ones and the 2 heavy black F4.0.
- **uscg_motorpool** (USCG motor pool shop building: foundation plan with footing schedule): Box the printed 15'-0" bay D-E on the top dimension string (label: scale: printed 15'-0" bay D-E), then the 79'-8" overall as a check. Read the FOOTING SCHEDULE at the top right: F60 is 6'-0" square, F66 6'-6", F70 7'-0", F80 8'-0", F96 9'-6" and F126 is 12'-6" x 9'-6". Box several of the scheduled footings (label: footing) and compare your measured area with the schedule size. The two X-crossed pits (label: pit) are in the key as well, but they are only about 40 px across on this sheet. Measure one and watch what the warning says. Box ONE footing with the label 'example: footing' and SAM 3 counts them all (there are 20), then do the same for the grid bubbles with the label 'example: grid bubble' (there are 32).
- **uscg_pile** (USCG bowling facility: pile footing plan (counting only)): Box the printed 16'-0" bay PF20-PF21 on the bottom dimension string (label: scale: printed 16'-0" bay PF20-PF21), and the 164'-0" overall as a check. Box ONE pile cap with the label 'example: pile footing' and SAM 3 counts them all. There are 29 and they are numbered PF1 to PF29, so you can check yourself. There is no area task on this sheet, on purpose. Each cap is 2'-6" x 5'-0" (read the PILE FOOTING SCHEDULE at the bottom left) and is drawn only 27 x 53 px, well under the ~60 px SAM 3 needs. Every attempt in the feasibility test came out 12 % to 35 % too big, because below that size the mask stops following the drawn rectangle and becomes a rounded copy of the box you drew. Take the 12.5 sq ft from the schedule instead.

In [ ]:
#@title ▶ Step 2b · Take-off on a structural plan { display-mode: "form" }
#@markdown Do **every** drawing in this list, one at a time: test_fp, test_fp_2, uscg_motorpool, uscg_pile. Zoom with the mouse wheel. Copy each table into your report.
drawing = "test_fp: Foundation plan: 12 ft square spread footings around an equipment pit" #@param ["test_fp: Foundation plan: 12 ft square spread footings around an equipment pit", "test_fp_2: Foundation plan with square spread footings and an elevator shaft", "uscg_motorpool: USCG motor pool shop building: foundation plan with footing schedule", "uscg_pile: USCG bowling facility: pile footing plan (counting only)"]
lab.takeoff(drawing)


> ### 📝 Report question 2
> From the structural plans: for each sheet, the scale and how you set it, the areas of the footings you measured against the sizes their marks give, and the count SAM 3 made from your one *example: footing* box (found / missed / extra). One sheet asks you only to count, because its pile caps are too small to measure - what happens to the measured area of something that small, and why?

### Step 2c · MEP plans

On an MEP sheet almost nothing is measured and almost everything is **counted**. The trade words - *light fixture*, *diffuser*, *sprinkler* - return nothing at all from the model, so counting is done the other way round: box **one** example of the symbol, labelled *example: 2x4 light fixture*, and the model finds every other symbol like it. Pick an example that is clean and lying the same way as most of the others; a rotated example loses about 40 % of the count.

- **va_ceiling** (VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)): There is no printed dimension on this sheet. Box ten cells of the 2 ft x 2 ft ceiling grid (label: scale: ten ceiling grid cells (2 ft each) = 20 ft) - that is 20 ft. Box the 0-16 ft graphic bar at the top right as well (label: scale: 0-16 ft graphic scale bar) and compare. One of the two is wrong by a factor of 2. Box ONE 2x4 light fixture - a horizontal one, in a clear part of the ceiling - with the label 'example: 2x4 light fixture' and SAM 3 counts them all. Do the same for the 2x2 light fixtures (label: example: 2x2 light fixture) and for the small round recessed lights (label: example: recessed light). There are no air diffusers, no return registers and no sprinkler heads anywhere on this sheet. Check the legend before you go looking for them.

In [ ]:
#@title ▶ Step 2c · Take-off on an MEP sheet { display-mode: "form" }
#@markdown Do **every** drawing in this list, one at a time: va_ceiling. Zoom with the mouse wheel. Copy each table into your report.
drawing = "va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)" #@param ["va_ceiling: VA outpatient clinic, lease module: reflected ceiling plan (light fixture count)"]
lab.takeoff(drawing)


> ### 📝 Report question 3
> From the ceiling plan: the count of each symbol that SAM 3 made from your one example box (found / missed / extra), the confidence you used, and what the extras were. Try a second example box of the same symbol, one that is rotated or sits in a cluttered spot, and report how the count changes. Why do the words *light fixture* and *diffuser* return nothing?

## Part 3 · Your own drawing

In [ ]:
#@title ▶ Your drawing, your words { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (it works on a phone too). Upload a drawing of your own, type what to find, move the threshold.
#@markdown To get square feet, measure a printed dimension on your sheet first: count the pixels along it, divide by its length in feet, and type that number in. Test at least three drawings of your own, from at least two disciplines, and take screenshots. Needs the live model.
lab.upload_app()


> ### 📝 Report question 4
> Test three drawings of your own, from at least two disciplines (a floor plan, a structural plan, an MEP sheet, a section, a site plan...). For each: the phrase or box you used, what came back, and whether it is right. Which kind of drawing failed, and can you say why?

> ### 📝 Report question 5
> Across the three disciplines: which take-off task was the most reliable and which the least, and what decides that (the size of the thing on the sheet, how often it repeats, whether it is drawn as a simple outline)? Where would you use this in practice, where would it mislead you, and what would you insist on having (a known dimension, a schedule, a second pair of eyes) before putting these numbers in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
#@markdown Every take-off you submitted, printed again in one place, plus the drawings you have not done yet.
lab.report_summary()


### Where the drawings come from, and the model

- **usda_5542** - USDA design 710-5542, 'Five-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 15. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **va_floor** - Outpatient / PACT Clinic, lease module (CBOC-L.pdf), VA Technical Information Library room templates. US Department of Veterans Affairs, Office of Construction & Facilities Management (2018). Public domain (work of the US federal government). https://www.cfm.va.gov/til/rTemplate/documents/CBOC-L.pdf
- **va_ceiling** - Outpatient / PACT Clinic, lease module (CBOC-L.pdf), VA Technical Information Library room templates. US Department of Veterans Affairs, Office of Construction & Facilities Management (2018). Public domain (work of the US federal government). https://www.cfm.va.gov/til/rTemplate/documents/CBOC-L.pdf
- **test_fp** - Foundation plan, example image 1 (test_fp.png), cropped to the two interior footing rows. Course example drawing (GitHub repository Haolan-Zhang/SAM_Example_Image). Provided for this course. https://github.com/Haolan-Zhang/SAM_Example_Image
- **test_fp_2** - Foundation plan, example image 2 (test_fp_2.png). Course example drawing (GitHub repository Haolan-Zhang/SAM_Example_Image). Provided for this course. https://github.com/Haolan-Zhang/SAM_Example_Image
- **uscg_motorpool** - Motor Pool Facility, Support Center New York, Shop Building (Building 928): structural foundation plan, details and notes, sheet S-1, 20 June 1984. U.S. Coast Guard, 3rd District Governors Island NY, Civil Engineering; drawn by Gonchor & Sput, Architects & Planners. Public domain (work of the US federal government). https://commons.wikimedia.org/wiki/File:Building_928_Structural_Foundation_Plan,_Details_and_Notes_Shop_Building,_June_20,_1984_-_DPLA_-_2938a6cfb0663504470f4b44fe7ae27e.tiff
- **uscg_pile** - Building 785, Sixteen Lane Bowling Facility, Governors Island: foundation plan, sheet 7 of 11, 19 January 1983. U.S. Coast Guard Support Center New York, Public Works Engineering (traced from International Design Consultants Inc.). Public domain (work of the US federal government). https://commons.wikimedia.org/wiki/File:Building_785_Sixteen_Lane_Bowling_Facility_Foundation_Plan,_January_19,_1983_-_DPLA_-_ec587aa2a393d5917cb73be19a2da195.tiff
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).